# London Transport Network 

Models the London contactless network as a graph to find the most likely interchange points between DLR and other services based on shortest journey time. 

### Key Concepts:
* **NLC (National Location Code):** Represents a set of physical ticket gates. Separates multi-modal stations (e.g., London Bridge NR vs. Underground) into distinct graph nodes.

* **ICS:** TfL’s station-level identifier used for Journey Planner API lookups.

* **NaPTAN (National Public Transport Access Node)**: A national standard system identifier used across the UK transport network to provide precise station-level granularity. The code is typically prefixed with 940GZZLU for Tube, 940GZZDL for DLR, or 910G/490G for National Rail stations.

**Output:** A comprehensive mapping table of all DLR-to-non-DLR journeys with optimal in-station interchange points.

In [1]:
# ==============================================================================
# LIBRARY IMPORTS & ENVIRONMENT SETUP
# ==============================================================================

# File management, environment variables, and graph serialization
import os
import pickle
from dotenv import load_dotenv

# Tabular data manipulation and numerical operations
import numpy as np
import pandas as pd

# Python built-ins for data processing and rate limiting
from itertools import combinations
import json
import re
import time

# Querying TfL API endpoints
import requests
import urllib

# Calculating distance between two stations
%pip install geopy
import geopy.distance

# Constructing network topology and running Dijkstra shortest path algotrithm
import networkx as nx

# Load API key from environment (.env file)
load_dotenv()
TFL_API_KEY = os.getenv("TFL_API_KEY", "")

if not TFL_API_KEY:
  print(
      "Warning: TFL_API_KEY not found in environment variables. API calls will"
      " rely on unauthenticated rate limits."
  )
else:
  print("TFL_API_KEY successfully loaded.")


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
TFL_API_KEY successfully loaded.


## Network Nodes - All Contactless Stations

Loads all contactless train stations across National Rail, Tube, DLR, Overground, and Elizabeth line. <br>
_Bus, tram, and river services are excluded._

Each station row maps its unique **NLC** (gate line) to its corresponding **ICS** station identifier.

In [2]:
# loading the station reference table
all_stations = pd.read_csv("all_stations_ref.csv")

# operators that are either already considered or are not accepting contactlessz payments
excluded_lines = [
    # already considered as separate lines
    "london-overground",
    "tfl-rail",
    # not contactless
    "cross-country",
    "virgin-trains-east-coast",
    "grand-central",
    "london-north-eastern-railway",
    "virgin-trains",
    "first-hull-trains",
]

#excluding selected lines from the station reference table
all_stations = all_stations[~all_stations["Line"].isin(excluded_lines)].copy()

# creating a list of NaPTAN and Gate_NLC codes, since there can be more than one code per station/line/NLC combination
def parse_list_field(val):
    if pd.isna(val) or str(val).lower() == "nan" or str(val).strip() == "":
        return []
    return [item.strip() for item in str(val).split(",") if item.strip()]

all_stations["NaPTAN"] = all_stations["NaPTAN"].apply(parse_list_field)
all_stations["Gate_NLC"] = all_stations["Gate_NLC"].apply(parse_list_field)

# grouping table to get distinct station/line/NLC combinations
all_stations = all_stations.groupby(["Station", "Line", "ICS", "NLC"]).agg({
    "NaPTAN": lambda x: list(set([item for sublist in x for item in sublist])),
    "Gate_NLC": lambda x: list(set([item for sublist in x for item in sublist]))
}).reset_index()

# creating a new Node_ID column that uniquely identifies each network node
all_stations["Node_ID"] = (
    all_stations["Station"].str.lower() + "_" + 
    all_stations["Line"].str.lower() + "_" + 
    all_stations["NLC"].astype(str)
)

# outputting the number of unique station/line/NLC nodes in the reference table
print(
    f"Loaded {all_stations['Node_ID'].nunique():,} unique station/line/NLC pairings, "
    f"with {all_stations['Station'].nunique():,} individual stations."
)

# saving the cleaned station reference table
all_stations.to_csv("all_stations_ref_cleaned.csv", index=False)

display(all_stations)

Loaded 1,233 unique station/line/NLC pairings, with 807 individual stations.


,Station,Line,ICS,NLC,NaPTAN,Gate_NLC,Node_ID
0,Abbey Road,dlr,1003006,750,[940GZZDLABR],[750],abbey road_dlr_750
1,Abbey Wood,elizabeth,1001001,5131,"[9100ABWD0, 910GABWDXR]",[5131],abbey wood_elizabeth_5131
2,Abbey Wood,southeastern,1001001,5131,[910GABWD],[5131],abbey wood_southeastern_5131
3,Abbey Wood,thameslink,1001001,5131,"[9100ABWD0, 910GABWD]",[5131],abbey wood_thameslink_5131
4,Acton Central,mildmay,1001002,1404,[910GACTNCTL],[1404],acton central_mildmay_1404
...,...,...,...,...,...,...,...
1230,Woolwich Dockyard,southeastern,1001345,5153,[910GWOLWCDY],[5153],woolwich dockyard_southeastern_5153
1231,Woolwich Dockyard,thameslink,1001345,5153,"[910GWOLWCDY, 9100WOLWCDY0]",[5153],woolwich dockyard_thameslink_5153
1232,Worcester Park,south-western-railway,1001346,5579,[910GWRCSTRP],[5579],worcester park_south-western-railway_5579
1233,Worplesdon,south-western-railway,90008069,5686,[910GWRPLSDN],[5686],worplesdon_south-western-railway_5686


In [5]:
ref_nlc = pd.read_csv("ref_nlc.csv")

nlc_col_list = all_stations["NLC"].values.tolist()
gate_nlc_col_list = sum(all_stations["Gate_NLC"].apply(lambda x: [i.replace("'", "") for i in x]), []) 
all_station_nlcs = nlc_col_list + gate_nlc_col_list

unique_nlc_list = list(set(map(int, all_station_nlcs))) # converting list to unique list of integers

# if NLC in ref_nlc is not in unique_list_int, then it is missing from the station reference table
missing_nlcs = ref_nlc[~ref_nlc["NLC"].isin(unique_nlc_list)]

# Luton airport DART stations are intentionally missing from the reference table. 
dart_stations = [3667, 9227]
missing_nlcs = missing_nlcs[~missing_nlcs["NLC"].isin(dart_stations)]

if missing_nlcs.shape[0] == 0:
    print("No new NLCs need to be added to the station reference table.")
else:
    print(f"Found {missing_nlcs.shape[0]:,} active NLCs")
    display(missing_nlcs)

No new NLCs need to be added to the station reference table.


## Network Edges: Adjacent Station Travel Times

### Data Source & Methodology:
* **Source:** Derived from TfL Line Route Sequence feeds and queried via the TfL Journey Planner API for active rail modes (Tube, DLR, Overground, Elizabeth Line and National Rail).

* **Assumptions & Fallbacks:** 
  * Only adjacent station pairs along active branch sequences are queried.
  * If a live API query fails or returns zero duration, a fallback travel time of `0.5` minutes (`FALLBACK_TRAVEL_TIME`) is applied and logged to `travel_times_investigation_log.csv`.

In [6]:
# ==============================================================================
# LINE TRAVEL TIMES BUILDER
# ==============================================================================

default_travel_time = 3

def api_query(a, b):
    jp_url = f"https://api.tfl.gov.uk/Journey/JourneyResults/{urllib.parse.quote(str(a))}/to/{urllib.parse.quote(str(b))}"

    try:
        response = requests.get(jp_url, params=TFL_API_KEY, timeout=4)
        if response.status_code == 200:
            return response.json()
        else:
            return None
    except requests.exceptions.RequestException:
        return None
    
def get_travel_times():
    travel_times_records = []
    station_locations_records = []

    lines = sorted(all_stations["Line"].dropna().unique().tolist())

    for line in lines:
        print(f"Processing line: {line}")
        
        line_seq_url = f"https://api.tfl.gov.uk/Line/{line}/Route/Sequence/all"
        line_seq_result = requests.get(line_seq_url, params=TFL_API_KEY)

        if line_seq_result.status_code != 200:
            continue

        line_seq_data = line_seq_result.json()

        for branch in line_seq_data.get("stopPointSequences", []):
            stops = branch.get("stopPoint", [])

            for i in range(len(stops) - 1):
                from_stop, to_stop = stops[i], stops[i + 1]

                from_naptan, to_naptan = from_stop.get("id", ""), to_stop.get("id", "")
                from_ics, to_ics = from_stop.get("icsId", ""), to_stop.get("icsId", "")

                # make sure both stops are in the all station ref file
                from_station_info = all_stations[
                    (all_stations["Line"].str.lower() == line.lower()) & 
                    ((all_stations["NaPTAN"].apply(lambda x: from_naptan in x)) |
                    (all_stations["ICS"] == from_ics))]
            
                to_station_info = all_stations[
                    (all_stations["Line"].str.lower() == line.lower()) &
                    ((all_stations["NaPTAN"].apply(lambda x: to_naptan in x)) |
                    (all_stations["ICS"] == to_ics))]

                if from_station_info.empty or to_station_info.empty:
                    continue

                # get location data to work out distance traveled
                to_lat, to_lon = to_stop.get("lat", None), to_stop.get("lon", None)
                from_lat, from_lon = from_stop.get("lat", None), from_stop.get("lon", None)
                
                if from_lat and from_lon and to_lat and to_lon:
                    distance = geopy.distance.geodesic((from_lat, from_lon), (to_lat, to_lon)).km
                else:
                    distance = None

                # API query ---
                duration = None
                status = None

                # NaPTAN code API query
                jp_naptan_result = api_query(from_naptan, to_naptan)

                if jp_naptan_result and len(jp_naptan_result.get("journeys", [])) > 0:
                    duration = jp_naptan_result["journeys"][0]["duration"] # Already in minutes!
                    status = "Success using NaPTAN"
                else: 
                    # ICS codes API query
                    jp_ics_result = api_query(from_ics, to_ics)

                    if jp_ics_result and len(jp_ics_result.get("journeys", [])) > 0:
                        duration = jp_ics_result["journeys"][0]["duration"]
                        status = "Success using ICS"
                    else:
                        # NaPTAN codes API query in reverse direction
                        jp_naptan_result = api_query(to_naptan, from_naptan)
                        if jp_naptan_result and len(jp_naptan_result.get("journeys", [])) > 0:
                            duration = jp_naptan_result["journeys"][0]["duration"]
                            status = "Success using NaPTAN in reverse"
                        else:
                            # ICS code API query in reverse direction
                            jp_ics_result = api_query(to_ics, from_ics)
                            if jp_ics_result and len(jp_ics_result.get("journeys", [])) > 0:
                                duration = jp_ics_result["journeys"][0]["duration"]
                                status = "Success using ICS in reverse"
                            else:
                                duration = None
                                status = "Average, API call failed"

                travel_times_records.append({
                    "line": line,
                    "from_station": from_station_info["Station"].values[0],
                    "to_station": to_station_info["Station"].values[0],
                    "from_node": from_station_info["Node_ID"].values[0],
                    "to_node": to_station_info["Node_ID"].values[0],
                    "distance_km": distance,
                    "duration_min": float(duration) if duration is not None else None,
                    "status": status
                })

                station_locations_records.append({
                    "line": line,
                    "station": from_station_info["Station"].values[0],
                    "Node_ID": from_station_info["Node_ID"].values[0],
                    "lat": from_lat,
                    "lon": from_lon
                })

    travel_times = pd.DataFrame(travel_times_records)
    station_location = pd.DataFrame(station_locations_records).drop_duplicates(subset=["Node_ID"])

    # Fill missing travel times with line average or default
    mean_line_duration = travel_times.groupby("line")["duration_min"].transform("mean")
    travel_times["duration_min"] = (
        travel_times["duration_min"]
        .fillna(mean_line_duration)
        .fillna(default_travel_time)
    ).round(1)

    return travel_times, station_location

travel_times, station_location = get_travel_times()

travel_times.to_csv("travel_times.csv", index=False)
station_location.to_csv("station_location.csv", index=False)

travel_times.head()

Processing line: bakerloo
Processing line: c2c
Processing line: central
Processing line: chiltern-railways
Processing line: circle
Processing line: district
Processing line: dlr
Processing line: east-midlands-railway
Processing line: east-midlands-trains
Processing line: elizabeth
Processing line: gatwick-express
Processing line: great-northern
Processing line: great-western-railway
Processing line: greater-anglia
Processing line: hammersmith-city
Processing line: heathrow-express
Processing line: jubilee
Processing line: liberty
Processing line: lioness
Processing line: london-midland
Processing line: metropolitan
Processing line: mildmay
Processing line: northern
Processing line: piccadilly
Processing line: south-western-railway
Processing line: southeastern
Processing line: southern
Processing line: suffragette
Processing line: thameslink
Processing line: victoria
Processing line: waterloo-city
Processing line: weaver
Processing line: west-midlands-trains
Processing line: windrush


,line,from_station,to_station,from_node,to_node,distance_km,duration_min,status
0,bakerloo,Harrow & Wealdstone,Kenton,harrow & wealdstone_bakerloo_597,kenton_bakerloo_620,1.725627,2.0,Success using NaPTAN
1,bakerloo,Kenton,South Kenton,kenton_bakerloo_620,south kenton_bakerloo_709,1.410414,2.0,Success using NaPTAN
2,bakerloo,South Kenton,North Wembley,south kenton_bakerloo_709,north wembley_bakerloo_659,0.908181,0.0,Success using NaPTAN
3,bakerloo,North Wembley,Wembley Central,north wembley_bakerloo_659,wembley central_bakerloo_751,1.243187,2.0,Success using NaPTAN
4,bakerloo,Wembley Central,Stonebridge Park,wembley central_bakerloo_751,stonebridge park_bakerloo_717,1.725099,3.0,Success using NaPTAN


In [7]:
# ==============================================================================
# LINE CONNECTIVITY & BRANCH COVERAGE AUDIT
# ==============================================================================

# 1. Gather all travel nodes from the travel_times dataset
from_travel_nodes = travel_times["from_node"].dropna().tolist()
to_travel_nodes = travel_times["to_node"].dropna().tolist()

all_travel_nodes = from_travel_nodes + to_travel_nodes
unique_travel_node_list = list(set(all_travel_nodes))

# a. Check for isolated nodes (in reference table but missing from travel edges)
isolated_nodes = all_stations[~all_stations["Node_ID"].isin(unique_travel_node_list)]

print(f"Total reference nodes: {len(all_stations):,}")
print(f"Connected travel nodes: {len(unique_travel_node_list):,}")
print(f"Isolated nodes found: {len(isolated_nodes):,}")

if len(isolated_nodes) > 0:
    print("\nSample isolated nodes:")
    display(isolated_nodes[["Station", "Line", "Node_ID"]].head(10))

# b. Check in-degree vs. out-degree balance
in_counts = travel_times["to_node"].value_counts()
out_counts = travel_times["from_node"].value_counts()

degree_df = pd.DataFrame({
    "in_degree": in_counts, 
    "out_degree": out_counts
}).fillna(0)

degree_df["diff"] = degree_df["out_degree"] - degree_df["in_degree"]
unbalanced_nodes = degree_df[degree_df["diff"] != 0]

print(f"\nNodes with unbalanced in/out degree counts: {len(unbalanced_nodes):,}")
if len(unbalanced_nodes) > 0:
    display(unbalanced_nodes.head(10))


Total reference nodes: 1,235
Connected travel nodes: 689
Isolated nodes found: 544

Sample isolated nodes:


,Station,Line,Node_ID
3,Abbey Wood,thameslink,abbey wood_thameslink_5131
4,Acton Central,mildmay,acton central_mildmay_1404
5,Acton Main Line,elizabeth,acton main line_elizabeth_3000
7,Acton Town,district,acton town_district_500
8,Acton Town,piccadilly,acton town_piccadilly_500
12,Aldgate,metropolitan,aldgate_metropolitan_502
13,Aldgate East,district,aldgate east_district_503
16,Alexandra Palace,thameslink,alexandra palace_thameslink_6025
18,Alperton,piccadilly,alperton_piccadilly_505
19,Amersham,chiltern-railways,amersham_chiltern-railways_506



Nodes with unbalanced in/out degree counts: 40


,in_degree,out_degree,diff
barnes_south-western-railway_5551,9.0,8.0,-1.0
blackfriars_southeastern_5112,1.0,2.0,1.0
canary wharf_dlr_842,3.0,2.0,-1.0
cannon street_southeastern_5142,7.0,9.0,2.0
chislehurst_southeastern_5099,5.0,6.0,1.0
clapham junction_south-western-railway_5595,28.0,27.0,-1.0
clapham junction_southern_5595,22.0,21.0,-1.0
east croydon_southern_5355,19.0,18.0,-1.0
ebbsfleet international_southeastern_5566,6.0,5.0,-1.0
feltham_south-western-railway_5559,2.0,3.0,1.0


## Network Edges: Expected Line Wait Times

### Data Source & Methodology:
* **Source:** Extracted dynamically using the TfL Timetable API (`/Line/{id}/Timetable/{stop}`) at designated benchmark hub stations (`LINE_BENCHMARK_STATIONS`).

* **Assumptions & Fallbacks:** 
  * Passenger arrivals are assumed to be uniform and random.
  * Expected passenger wait time is computed using the half-headway rule: $\text{Wait} = \frac{60}{2 \times \text{TPH}}$ (Trains Per Hour), measuring peak arrivals at hour 08:00–09:00 and off-peak at 11:00–12:00.
  * Where timetable feeds are unavailable or fail, rule-based mode-level headways are applied as explicit fallbacks (e.g. Tube/DLR: 2.5m, Elizabeth Line: 2.0m, Overground: 5.0m, National Rail: 7.5m).

In [ ]:
# ==============================================================================
# LINE WAIT TIMES
# ==============================================================================

fallback = {
    "tube": 2.5,
    "dlr": 2.5,
    "overground": 5.0,
    "elizabeth-line": 2.0,
    "national-rail": 7.5
}

wait_times = []

def get_station_wait_time(line_id: str, station: str, node: str, stop_naptans: list):

    # Fallback default if line type isn't matched
    default_fallback = 3.0
    line_key = str(line_id).lower()
    fallback_wait = next((v for k, v in fallback.items() if k in line_key), default_fallback)

    # Ensure we have a valid NaPTAN string from the list
    if not stop_naptans or not isinstance(stop_naptans, list):
        wait_times.append({
            "Line": line_id, "Station": station, "Node_ID": node,
            "NaPTAN": None, "Wait_Time": fallback_wait
        })
        return

    stop_naptan = stop_naptans[0]
    url = f"https://api.tfl.gov.uk/Line/{line_id}/Timetable/{stop_naptan}?direction=outbound"
    
    try:
        res = requests.get(url, params=TFL_API_KEY, timeout=5)
    except requests.exceptions.RequestException:
        res = None

    working_hours = {}

    if res and res.status_code == 200:
        data = res.json()
        routes = data.get("timetable", {}).get("routes", [])

        for route in routes:
            for schedule in route.get("schedules", []):
                valid_days = schedule.get("validDays", [])
                
                if "Mon" in str(valid_days) or "Monday" in str(valid_days) or len(schedule.get("knownJourneys", [])) > 500:
                    for entry in schedule.get("knownJourneys", []):
                        hour = int(entry.get("hour", 0))
                        if 6 < hour < 24:
                            working_hours[hour] = working_hours.get(hour, 0) + 1
                    break

    active_hours = [h for h in range(7, 24) if h in working_hours]

    if active_hours:
        total_trains = sum(working_hours[h] for h in active_hours)
        num_hours = len(active_hours)

        avg_trains_per_hour = total_trains / num_hours
        avg_wait_time = round(60 / (2 * avg_trains_per_hour), 1)

        wait_times.append({
            "Line": line_id,
            "Station": station,
            "Node_ID": node,
            "NaPTAN": stop_naptan,
            "Wait_Time": avg_wait_time
        })
    else:
        # Use fallback if API fails or returns no active hours
        wait_times.append({
            "Line": line_id,
            "Station": station,
            "Node_ID": node,
            "NaPTAN": stop_naptan,
            "Wait_Time": fallback_wait
        })

# get wait time for each station 
for _, station in all_stations.iterrows():
    get_station_wait_time(
        station["Line"], 
        station["Station"], 
        station["Node_ID"], 
        station["NaPTAN"]
    )

# convert results into a table
wait_times = pd.DataFrame(wait_times)

wait_times.to_csv("line_wait_times.csv", index=False)
display(wait_times.head())

## Network Edges: Interchange Walking Penalties

### Data Source & Methodology:
* **Source:** Combines published TfL Pay As You Go Out-of-Station Interchange (OSI) timeout matrix (`osi_matrix.csv`) with synthetic API journey triads for in-station platform transfers.

* **Assumptions & Fallbacks:** 
  * OSI fare timeout limits are mapped to physical walking durations ($\le 10\text{m} \to 4.0\text{m}$, $\le 15\text{m} \to 5.0\text{m}$, etc.).

  * **In-Station Interchanges:** Evaluated using a 4-tier hierarchy:
    1. *Empirical Layout Overrides* (e.g., Bank, King's Cross, Waterloo complex hubs).
    2. *Cross-Platform Benchmarks* (2.0m flat penalty for same-platform transfers).
    3. *Synthetic API Triads* (${\text{Walk} = \text{Journey}_{A \to C} - (\text{Leg}_A + \text{Leg}_B + \text{Wait}_B)}$).
    4. *Depth-Tier Heuristics* (Fallback rule based on surface, subsurface, and deep tube layering).

  * **OSI Exclusion:** Out-of-station interchanges will be skipped when building the strict in-station routing graph.

In [ ]:
import pandas as pd
import ast
from collections import defaultdict

default_interchange_time = 2.0

# 1. Parsing helper (reusing your safe parser)
def parse_set(val):
    if isinstance(val, (pd.Series, list)):
        val = list(val)
    elif val is None or pd.isna(val) or str(val).lower() == "nan":
        return set()
    if isinstance(val, list):
        return {str(x).strip().replace(".0", "") for x in val if pd.notna(x) and str(x).strip() and str(x).strip() != "nan"}
    try:
        evaluated = ast.literal_eval(str(val))
        if isinstance(evaluated, list):
            return {str(x).strip().replace(".0", "") for x in evaluated if pd.notna(x) and str(x).strip() and str(x).strip() != "nan"}
    except:
        pass
    return {x.strip().replace(".0", "") for x in str(val).split(",") if x.strip() and x.strip() != "nan"}

# 2. Build inverted indices (Gate/NaPTAN -> set of Node_IDs)
gate_to_nodes = defaultdict(set)
naptan_to_nodes = defaultdict(set)

for _, row in all_stations.iterrows():
    node_id = row["Node_ID"]
    for g in parse_set(row["Gate_NLC"]):
        gate_to_nodes[g].add(node_id)
    for n in parse_set(row["NaPTAN"]):
        naptan_to_nodes[n].add(node_id)

interchanges_dict = {}

# Helper to add or merge interchange pairs
def add_interchange(node_a, node_b, gate=None, naptan=None):
    if node_a == node_b:
        return
    pair = tuple(sorted([node_a, node_b]))
    if pair not in interchanges_dict:
        interchanges_dict[pair] = {
            "From_Node_ID": pair[0],
            "To_Node_ID": pair[1],
            "Shared_Gate_NLCs": set(),
            "Shared_NaPTANs": set(),
            "Interchange_Time_Min": default_interchange_time
        }
    if gate:
        interchanges_dict[pair]["Shared_Gate_NLCs"].add(gate)
    if naptan:
        interchanges_dict[pair]["Shared_NaPTANs"].add(naptan)
        # Shared platform NaPTAN overrides interchange time to 0.0
        interchanges_dict[pair]["Interchange_Time_Min"] = 0.0

# 3. Form pairs from shared gates
for gate, nodes in gate_to_nodes.items():
    if len(nodes) > 1:
        nodes_list = list(nodes)
        for i in range(len(nodes_list)):
            for j in range(i + 1, len(nodes_list)):
                add_interchange(nodes_list[i], nodes_list[j], gate=gate)

# 4. Form pairs from shared NaPTANs
for naptan, nodes in naptan_to_nodes.items():
    if len(nodes) > 1:
        nodes_list = list(nodes)
        for i in range(len(nodes_list)):
            for j in range(i + 1, len(nodes_list)):
                add_interchange(nodes_list[i], nodes_list[j], naptan=naptan)

# Convert dictionary back to a clean list/DataFrame
interchanges = []
for data in interchanges_dict.values():
    data["Shared_Gate_NLCs"] = list(data["Shared_Gate_NLCs"])
    data["Shared_NaPTANs"] = list(data["Shared_NaPTANs"])
    interchanges.append(data)

df_interchanges = pd.DataFrame(interchanges)
print(f"Total unique interchange pairs: {len(df_interchanges):,}")
display(df_interchanges.head())

In [ ]:
# 1. Filter for only default interchanges (behind-the-barrier, non-cross-platform)
default_interchanges = [item for item in interchanges if item["Interchange_Time_Min"] == 2.0]

# 2. Build directional lookups from travel_times for fast adjacency queries
# For any node, what are its outgoing and incoming neighbors + durations?
outgoing_travel = {} # from_node -> {to_node: duration_min}
incoming_travel = {} # to_node -> {from_node: duration_min}

for _, row in travel_times.dropna(subset=["from_node", "to_node"]).iterrows():
    f_node = row["from_node"]
    t_node = row["to_node"]
    dur = row["duration_min"]
    
    outgoing_travel.setdefault(f_node, {})[t_node] = dur
    incoming_travel.setdefault(t_node, {})[f_node] = dur

print(f"Analyzing {len(default_interchanges):,} default interchange pairs...")

# 3. Iterate through default interchanges
for interchange in default_interchanges:
    node_a = interchange["From_Node_ID"] # e.g., Line 1 station platform
    node_b = interchange["To_Node_ID"]   # e.g., Line 2 station platform
    
    # Find stations adjacent TO node_a (trains coming into A, or predecessors)
    # i.e., predecessors -> node_a
    predecessors_a = incoming_travel.get(node_a, {})
    
    # Find stations adjacent FROM node_b (trains leaving B, or successors)
    # i.e., node_b -> successors
    successors_b = outgoing_travel.get(node_b, {})
    
    # Alternatively, if you want paths passing through both or connecting them:
    # Let's check direct transit links or paths bridging through the interchange pair
    
    # Example diagnostic print for the first few
    print(f"\nInterchange: {node_a} <--> {node_b}")
    print(f"  - Predecessors feeding into A: {list(predecessors_a.keys())}")
    print(f"  - Successors leaving from B: {list(successors_b.keys())}")
    
    # To find multi-hop routes (e.g., Station X -> Node A -> Node B -> Station Y):
    # You can match incoming routes to A and outgoing routes from B
    calculated_path_durations = []
    for pred_a, dur_in in predecessors_a.items():
        for succ_b, dur_out in successors_b.items():
            total_duration = dur_in + interchange["Interchange_Time_Min"] + dur_out
            calculated_path_durations.append(total_duration)
            
    if calculated_path_durations:
        avg_travel_time = sum(calculated_path_durations) / len(calculated_path_durations)
        print(f"  - Found {len(calculated_path_durations)} connecting paths through this interchange.")
        print(f"  - Average composite travel time: {avg_travel_time:.1f} mins")
    
    break # Remove break once you want to inspect more than the first one

In [ ]:
# ==============================================================================
# BEHIND-THE-BARRIER INTERCHANGE BUILDER (MULTI-GATE SAFE)
# ==============================================================================
# Identifies valid interchange pairs between network nodes that share at 
# least one common Gate_NLC, handling comma-separated multi-gate entries safely.
# ==============================================================================

def build_barrier_interchanges(stations_df, default_interchange_time=2.0):
  """
    Computes bidirectional behind-the-barrier interchange edges by checking 
    shared Gate_NLCs across distinct composite Node_IDs.
  """
  print("Computing behind-the-barrier interchanges from shared Gate_NLCs...")

  # Ensure Node_ID exists in the dataframe
  if "Node_ID" not in stations_df.columns:
    stations_df["Node_ID"] = (
        stations_df["Station"].astype(str) + "_" + 
        stations_df["Line"].astype(str) + "_" + 
        stations_df["Gate_NLC"].astype(str)
    )

  # Robust function to parse single or comma-separated Gate_NLC entries into a set of integers
  def parse_gate_nlcs(series):
    gate_set = set()
    for val in series:
      if pd.notna(val):
        # Split by comma in case of multi-gate entries like '501, 577'
        parts = str(val).split(",")
        for p in parts:
          p_clean = p.strip()
          if p_clean.isdigit():
            gate_set.add(int(p_clean))
    return gate_set

  # Map each Node_ID to its set of Gate_NLC integers
  node_gates = (
      stations_df.groupby("Node_ID")["Gate_NLC"]
      .apply(parse_gate_nlcs)
      .to_dict()
  )

  nodes = list(node_gates.keys())
  interchanges = []

  # Compare all unique node pairs
  for i in range(len(nodes)):
    for j in range(i + 1, len(nodes)):
      node_a = nodes[i]
      node_b = nodes[j]

      common_gates = node_gates[node_a].intersection(node_gates[node_b])

      if common_gates:
        # Add forward edge (Node A -> Node B)
        interchanges.append({
            "From_Node_ID": node_a,
            "To_Node_ID": node_b,
            "Shared_Gate_NLCs": list(common_gates),
            "Interchange_Time_Mins": default_interchange_time,
            "Interchange_Type": "Behind Barrier"
        })
        # Add reverse edge (Node B -> Node A) for bidirectional graph routing
        interchanges.append({
            "From_Node_ID": node_b,
            "To_Node_ID": node_a,
            "Shared_Gate_NLCs": list(common_gates),
            "Interchange_Time_Mins": default_interchange_time,
            "Interchange_Type": "Behind Barrier"
        })

  df_interchanges = pd.DataFrame(interchanges)
  df_interchanges.to_csv("behind_barrier_interchanges.csv", index=False)

  print(f"Successfully generated {len(df_interchanges):,} directional behind-the-barrier interchange edges.")
  return df_interchanges

# Execute interchange builder using the loaded 'all_stations' dataframe
barrier_interchanges = build_barrier_interchanges(all_stations)
barrier_interchanges.head()

In [ ]:
barrier_interchanges.to_csv("interchanges.csv", index=False)

In [ ]:
# ==============================================================================
# SYSTEMATIC INTERCHANGE WALTING PENALTIES & CLASSIFICATION BUILDER
# ==============================================================================
# Categorizes station transfers into Behind-the-Gate In-Station Interchanges, 
# Gated OSIs, and Ungated OSIs using NLC counts and GATED status from all_stations.csv.
# ==============================================================================


def convert_osi_timeout_to_walk_mins(timeout_mins: float) -> float:
  """Converts TfL PAYG revenue timeout limits into physical walking durations."""
  timeout_mins = float(timeout_mins)
  if timeout_mins <= 10:
    return 4.0
  elif timeout_mins <= 15:
    return 5.0
  elif timeout_mins <= 20:
    return 7.0
  elif timeout_mins <= 30:
    return 8.5
  else:
    return 10.0


# Depth tier fallbacks for physical platform walking durations
LINE_DEPTH_TIERS = {
    "Surface/Rail": {
        "National Rail",
        "London Overground",
        "c2c",
        "Chiltern Railways",
        "Southeastern",
        "Thameslink",
        "DLR",
        "Great Northern",
        "Great Western Railway",
        "South Western Railway",
        "Lioness",
        "Mildmay",
        "Windrush",
        "Weaver",
        "Suffragette",
        "Liberty",
    },
    "Subsurface": {"Circle", "District", "Hammersmith & City", "Metropolitan"},
    "Deep Tube": {
        "Bakerloo",
        "Central",
        "Jubilee",
        "Northern",
        "Piccadilly",
        "Victoria",
        "Waterloo & City",
        "Elizabeth line",
    },
}


def get_line_tier(line_name: str) -> str:
  for tier, lines in LINE_DEPTH_TIERS.items():
    if line_name in lines:
      return tier
  return "Deep Tube"


def get_depth_tier_walk_mins(line_a: str, line_b: str) -> float:
  t_a, t_b = get_line_tier(line_a), get_line_tier(line_b)
  if t_a == t_b:
    return 2.0
  elif {t_a, t_b} == {"Surface/Rail", "Deep Tube"}:
    return 7.0
  elif {t_a, t_b} == {"Subsurface", "Deep Tube"}:
    return 5.0
  elif {t_a, t_b} == {"Surface/Rail", "Subsurface"}:
    return 4.0
  return 3.0


# Strict behind-the-gate DLR multi-line hub overrides (where distinct lines share NLCs)
STRICT_DLR_IN_STATION_HUBS = {
    513,  # Bank / Monument Unified
    719,  # Stratford
    884,  # Canning Town
    5145,  # Greenwich
}


def generate_systematic_interchanges():
  df_stations = all_stations.copy()
  df_stations["NLC_int"] = (
      pd.to_numeric(df_stations["NLC"], errors="coerce")
      .dropna()
      .astype(int)
  )

  # Station name NLC multiplicity map
  station_nlc_counts = df_stations.groupby("STATION")["NLC_int"].nunique().to_dict()

  # NLC Gated status map ("gated" vs "ungated")
  nlc_gated_map = dict(
      zip(
          df_stations["NLC_int"],
          df_stations["GATED"].str.strip().str.lower().fillna("gated"),
      )
  )

  nlc_name_map = dict(
      zip(df_stations["NLC_int"], df_stations["STATION"].str.strip())
  )

  df_travel = travel_times
  df_wait = wait_times
  df_osi = pd.read_csv("osi_matrix.csv")

  wait_dict = dict(
      zip(
          df_wait["Line"].str.strip(),
          df_wait["Default Avg Wait (mins)"].astype(float),
      )
  )

  records = []

  # STEP 1: Process Published Out-of-Station Interchanges (OSIs) from Report
  for _, row in df_osi.iterrows():
    try:
      nlc_a = int(row["Station_A"])
      nlc_b = int(row["Station_B"])

      if nlc_b == 0 or nlc_a == nlc_b:
        continue

      raw_timeout = float(row["InterchangeTimeAB"])
      walk_mins = convert_osi_timeout_to_walk_mins(raw_timeout)

      st_a = nlc_name_map.get(nlc_a, str(row["Station_A_Name"]).strip())
      st_b = nlc_name_map.get(nlc_b, str(row["Station_B_Name"]).strip())
      combined_name = f"{st_a} / {st_b}"

      lines_a = df_travel[
          (df_travel["From_NLC"] == nlc_a) | (df_travel["To_NLC"] == nlc_a)
      ]["Line"].unique()
      lines_b = df_travel[
          (df_travel["From_NLC"] == nlc_b) | (df_travel["To_NLC"] == nlc_b)
      ]["Line"].unique()

      gated_a = nlc_gated_map.get(nlc_a, "gated")
      gated_b = nlc_gated_map.get(nlc_b, "gated")
      ic_type = (
          "Out-of-Station Interchange (OSI)"
          if (gated_a == "gated" and gated_b == "gated")
          else "Ungated Out-of-Station Interchange"
      )

      for l_a in lines_a:
        for l_b in lines_b:
          records.append({
              "station": combined_name,
              "station_NLC": nlc_a,
              "From_Line": l_a.strip(),
              "To_Line": l_b.strip(),
              "walk_interchange_times": walk_mins,
              "interchange_type": ic_type,
              "source": "Published TfL OSI Matrix",
          })
    except Exception:
      continue

  # STEP 2: Systematic In-Station vs OSI Classification for Multi-Line Hubs
  from_df = df_travel[["From_NLC", "From_Station", "Line"]].rename(
      columns={"From_NLC": "NLC", "From_Station": "Station"}
  )
  to_df = df_travel[["To_NLC", "To_Station", "Line"]].rename(
      columns={"To_NLC": "NLC", "To_Station": "Station"}
  )
  station_lines = pd.concat([from_df, to_df]).drop_duplicates()

  multi_line_hubs = (
      station_lines.groupby("NLC")["Line"].nunique().loc[lambda x: x > 1].index
  )

  for nlc in multi_line_hubs:
    sub = station_lines[station_lines["NLC"] == nlc]
    st_name = sub["Station"].iloc[0].strip()
    lines = sub["Line"].tolist()
    unique_lines = list(set([l.strip() for l in lines]))

    mapped_nlc = NLC_ALIASES.get(int(nlc), int(nlc))
    num_nlcs_at_station = station_nlc_counts.get(st_name, 1)
    gated_status = nlc_gated_map.get(int(nlc), "gated")

    for i in range(len(unique_lines)):
      for j in range(i + 1, len(unique_lines)):
        l_a, l_b = unique_lines[i], unique_lines[j]

        # Systematic Classification Rule:
        # If single NLC station OR whitelisted behind-the-gate hub -> In-Station Interchange
        # Otherwise -> Out-of-Station Interchange (OSI) or Concourse Transfer
        is_dlr_involved = (
            "dlr" in l_a.lower()
            or "dlr" in l_b.lower()
            or "dlr" in st_name.lower()
        )

        if num_nlcs_at_station == 1 or mapped_nlc in STRICT_DLR_IN_STATION_HUBS:
          ic_type = "In-Station Interchange"
          source = "Systematic Rule (Single NLC / Gated Hub)"
          walk_mins = 2.0  # Default flat cross-platform walk
        elif gated_status == "ungated":
          ic_type = "Ungated Out-of-Station Interchange"
          source = "Systematic Rule (Ungated Node)"
          walk_mins = 4.0
        else:
          ic_type = "Out-of-Station Interchange (OSI)"
          source = "Systematic Rule (Multi-NLC Barrier Separation)"
          walk_mins = get_depth_tier_walk_mins(l_a, l_b)

        records.append({
            "station": st_name,
            "station_NLC": int(nlc),
            "From_Line": l_a,
            "To_Line": l_b,
            "walk_interchange_times": walk_mins,
            "interchange_type": ic_type,
            "source": source,
        })

  # STEP 3: Deduplicate & Enforce Bidirectional Symmetry
  df_out = pd.DataFrame(records).drop_duplicates(
      subset=["station_NLC", "From_Line", "To_Line"]
  )

  reverse_records = []
  for _, row in df_out.iterrows():
    reverse_records.append({
        "station": row["station"],
        "station_NLC": row["station_NLC"],
        "From_Line": row["To_Line"],
        "To_Line": row["From_Line"],
        "walk_interchange_times": row["walk_interchange_times"],
        "interchange_type": row["interchange_type"],
        "source": row["source"],
    })

  df_final = pd.concat([df_out, pd.DataFrame(reverse_records)]).drop_duplicates(
      subset=["station_NLC", "From_Line", "To_Line"]
  )

  cols = [
      "station",
      "station_NLC",
      "From_Line",
      "To_Line",
      "walk_interchange_times",
      "interchange_type",
      "source",
  ]
  interchange_times = df_final[cols].sort_values(
      by=["station", "From_Line", "To_Line"]
  )

  print(
      "Successfully generated"
      f" {len(interchange_times):,} systematically classified interchange"
      " edges."
  )
  print("\nBreakdown by Interchange Type:")
  print(interchange_times["interchange_type"].value_counts())

  return interchange_times

interchange_times = generate_systematic_interchanges()
interchange_times.head()

## Network Graph Construction (`NetworkX`) with NLC Aliasing

Constructs a directed graph representing the London contactless transit network:

* **NLC Aliasing:** Harmonizes legacy, variant, and adjacent station codes (e.g., Monument $\to$ Bank, Bow Road $\to$ Bow Church) using `NLC_ALIASES` to ensure methodological consistency.

* **Nodes:** Represent unique physical gate lines (`NLC_Line`), keeping multi-modal platforms (London Bridge NR vs. Underground) separate.

* **Edges (OSIs Excluded):** Journey travel times, boarding wait times, and in-station interchange walking penalties.

In [ ]:
# ==============================================================================
# NETWORK GRAPH CONSTRUCTION & DIJKSTRA TOPOLOGY BUILDER (WITH NLC ALIASES)
# ==============================================================================
# Constructs a directed NetworkX graph with NLC aliasing applied to nodes and edges.
# Out-of-Station Interchanges (OSIs) are strictly excluded.
# ==============================================================================

print("Initializing NetworkX Directed Graph...")
G = nx.DiGraph()

# Mapping of legacy/variant NLCs and adjacent station stops to primary DLR/hub NLCs
NLC_ALIASES = {
    # Legacy / Variant DLC & Rail NLC Unification
    843: 884,  # Canning Town DLR -> Canning Town Primary NLC
    573: 5152,  # Woolwich Arsenal DLR -> Woolwich Arsenal Primary NLC
    844: 513,  # Bank DLR -> Bank Primary NLC
    863: 719,  # Stratford DLR -> Stratford Primary NLC
    6960: 719,  # Stratford NR -> Stratford Primary NLC
    1082: 860,  # Shadwell LO -> Shadwell Primary NLC
    842: 852,  # Canary Wharf DLR -> Canary Wharf Primary NLC
    887: 6561,  # Custom House DLR -> Custom House Primary NLC
    928: 5145,  # Greenwich DLR -> Greenwich Primary NLC
    862: 7491,  # Limehouse DLR -> Limehouse Primary NLC
    940: 5051,  # Lewisham DLR -> Lewisham Primary NLC
    559: 889,  # West Silvertown DLR -> West Silvertown Primary NLC

    # Adjacent Station / Cross-Platform Mappings (Consistency with Manual Review)
    528: 851,  # Bow Road LU -> Bow Church DLR
    644: 513,  # Monument LU -> Bank Unified Hub
    731: 888,  # Tower Hill LU -> Tower Gateway DLR
}

# 1. Build lookup maps and apply NLC_ALIASES to station NLCs
df_stations = all_stations.copy()
df_stations["NLC_int"] = pd.to_numeric(
    df_stations["NLC"], errors="coerce"
).astype("Int64")

# Map original NLC to aliased primary NLC
nlc_to_name = {}
for _, row in df_stations.dropna(subset=["NLC_int"]).iterrows():
  orig_nlc = int(row["NLC_int"])
  mapped_nlc = NLC_ALIASES.get(orig_nlc, orig_nlc)
  station_name = str(row["STATION"]).strip()
  nlc_to_name[mapped_nlc] = station_name

# Identify DLR stations using mapped NLCs
dlr_stations_df = df_stations[
    df_stations["STATION"].str.contains("DLR", case=False, na=False)
]
dlr_nlcs = set(
    dlr_stations_df["NLC_int"]
    .dropna()
    .astype(int)
    .map(lambda x: NLC_ALIASES.get(x, x))
)

print(
    f"Loaded {len(nlc_to_name):,} unique station nodes ({len(dlr_nlcs)} DLR"
    " endpoint groups)."
)

# 2. Add Journey Edges (with NLC Aliasing applied)
travel_added = 0
for _, row in travel_times.iterrows():
  try:
    orig_from_nlc = int(row["From_NLC"])
    orig_to_nlc = int(row["To_NLC"])

    # Apply NLC Aliasing
    from_nlc = NLC_ALIASES.get(orig_from_nlc, orig_from_nlc)
    to_nlc = NLC_ALIASES.get(orig_to_nlc, orig_to_nlc)

    line = str(row["Line"]).strip()
    duration = float(row["Travel_Time_Mins"])

    suffix = "DLR" if line.lower() == "dlr" else line
    u_node = f"{from_nlc}_{suffix}"
    v_node = f"{to_nlc}_{suffix}"

    G.add_edge(u_node, v_node, weight=duration, edge_type="journey", line=line)
    G.add_edge(v_node, u_node, weight=duration, edge_type="journey", line=line)
    travel_added += 2
  except Exception:
    continue

print(f"Added {travel_added:,} directional journey edges.")

# 3. Add Wait Edges (Initial boarding wait times)
wait_dict = dict(
    zip(
        wait_times["Line"].str.strip(),
        wait_times["Default Avg Wait (mins)"].astype(float),
    )
)
wait_added = 0
for nlc in nlc_to_name.keys():
  station_lines = travel_times[
      (travel_times["From_NLC"].map(lambda x: NLC_ALIASES.get(int(x), int(x)))
       == nlc)
      | (
          travel_times["To_NLC"].map(
              lambda x: NLC_ALIASES.get(int(x), int(x))
          )
          == nlc
      )
  ]["Line"].unique()

  for line in station_lines:
    suffix = "DLR" if str(line).lower() == "dlr" else str(line)
    node_id = f"{nlc}_{suffix}"
    if G.has_node(node_id):
      wait_time = wait_dict.get(str(line).strip(), 2.5)
      G.add_edge(
          node_id, node_id, weight=wait_time, edge_type="wait", line=line
      )
      wait_added += 1

print(f"Added {wait_added:,} station boarding wait edges.")

# 4. Add Interchange Edges (In-station platform transfers, OSIs strictly excluded)
interchange_added = 0
for _, row in interchange_times.iterrows():
  try:
    if "OSI" in str(row["interchange_type"]):
      continue  # Skip OSIs

    orig_nlc = int(row["station_NLC"])
    nlc = NLC_ALIASES.get(orig_nlc, orig_nlc)

    l_from = str(row["From_Line"]).strip()
    l_to = str(row["To_Line"]).strip()
    walk_time = float(row["walk_interchange_times"])

    s_from = "DLR" if l_from.lower() == "dlr" else l_from
    s_to = "DLR" if l_to.lower() == "dlr" else l_to

    u_node = f"{nlc}_{s_from}"
    v_node = f"{nlc}_{s_to}"

    if G.has_node(u_node) and G.has_node(v_node):
      G.add_edge(
          u_node,
          v_node,
          weight=walk_time,
          edge_type="interchange",
          line=f"{l_from}->{l_to}",
      )
      interchange_added += 1
  except Exception:
    continue

print(
    "Added"
    f" {interchange_added:,} in-station interchange edges (OSIs excluded)."
)
print(
    "\nGraph Construction Complete: Total Nodes ="
    f" {len(G.nodes):,} | Total Edges = {len(G.edges):,}"
)

## Serialize & Save Network Graph

Saves the in-memory NetworkX directed graph and associated lookup dictionaries (`nlc_to_name`, `df_stations`) to disk (`network_graph_no_osi.pkl`). This allows subsequent routing and auditing cells to load the network topology instantly without rebuilding it.

In [ ]:
# ==============================================================================
# SERIALIZE & SAVE NETWORK GRAPH
# ==============================================================================
# Packages the NetworkX graph and lookup metadata into a dictionary and 
# serializes it using pickle for downstream routing scripts and notebooks.
# ==============================================================================

# Bundle graph and metadata structures
graph_data = {
    "graph": G,
    "nlc_to_name": nlc_to_name,
    "stations": df_stations,
}

output_filename = "network_graph.pkl"

with open(output_filename, "wb") as f:
  pickle.dump(graph_data, f)

print(
    f"Successfully serialized and saved network graph to '{output_filename}'"
    f" ({os.path.getsize(output_filename) / (1024 * 1024):.2f} MB)."
)

## DLR Routing Engine

Executes Dijkstra shortest-path routing across the network graph from all non-DLR station gate lines to all DLR station endpoints (and vice versa):

* **In-Gate Routing Only:** OSIs are excluded, ensuring only valid platform-to-platform transfers are used.

* **DLR Interchange Identification:** Traces each shortest path to identify the exact node where the journey transitions onto or off the DLR network.

* **Output Schema:** Produces a comprehensive CSV table (`dlr_journey_interchanges.csv`) containing origin/destination names, NLCs, designated DLR interchanges, full interchange station lists, and formatted journey times.

In [ ]:
# ==============================================================================
# BATCH DLR ROUTING ENGINE & INTERCHANGE AUDIT BUILDER
# ==============================================================================
# Computes shortest paths between all non-DLR and DLR NLC gate lines,
# extracts transfer transition points, and formats the output into the 
# required 8-column schema.
# ==============================================================================


def get_route_interchanges_and_dlr_point(path, nlc_to_name):
  """Traces a NetworkX path to identify all interchange stations

  and the specific DLR interchange transition point.
  """
  dlr_interchange_nlc = None
  dlr_interchange_name = None
  all_interchanges = []

  for i in range(len(path) - 1):
    u_node, v_node = path[i], path[i + 1]
    u_parts = u_node.split("_")
    v_parts = v_node.split("_")

    u_nlc, u_line = int(u_parts[0]), u_parts[1]
    v_nlc, v_line = int(v_parts[0]), v_parts[1]

    # Detect line transition (Interchange)
    if u_line != v_line and u_nlc == v_nlc:
      mapped_nlc = NLC_ALIASES.get(u_nlc, u_nlc)
      st_name = nlc_to_name.get(mapped_nlc, f"NLC_{mapped_nlc}")
      if st_name not in all_interchanges:
        all_interchanges.append(st_name)

    # Detect DLR transition (Onto or off DLR)
    if (u_line == "DLR" and v_line != "DLR") or (
        u_line != "DLR" and v_line == "DLR"
    ):
      transition_nlc = u_nlc if u_line == "DLR" else v_nlc
      mapped_trans_nlc = NLC_ALIASES.get(transition_nlc, transition_nlc)
      dlr_interchange_nlc = mapped_trans_nlc
      dlr_interchange_name = nlc_to_name.get(
          mapped_trans_nlc, f"NLC_{mapped_trans_nlc}"
      )

  # If route is entirely DLR or direct without explicit boundary node capture
  if not dlr_interchange_name and path:
    for node in path:
      parts = node.split("_")
      if parts[1] == "DLR":
        raw_nlc = int(parts[0])
        mapped_nlc = NLC_ALIASES.get(raw_nlc, raw_nlc)
        dlr_interchange_nlc = mapped_nlc
        dlr_interchange_name = nlc_to_name.get(
            mapped_nlc, f"NLC_{mapped_nlc}"
        )
        break

  all_interchanges_str = (
      ", ".join(all_interchanges) if all_interchanges else "Direct / None"
  )
  return dlr_interchange_nlc, dlr_interchange_name, all_interchanges_str


def run_batch_dlr_routing():
  # Identify DLR nodes and NLCs
  dlr_nodes = [n for n in G.nodes if n.endswith("_DLR")]
  dlr_nlcs = set(int(n.split("_")[0]) for n in dlr_nodes)
  all_nlcs = set(df_stations["NLC"].dropna().astype(int))
  non_dlr_nlcs = all_nlcs - dlr_nlcs

  print(
      f"Running batch routing across {len(dlr_nlcs)} DLR endpoint NLCs and"
      f" {len(non_dlr_nlcs)} non-DLR NLC gate lines..."
  )
  results = []

  # Helper to process and append route results
  def add_route_record(direction, u_nlc, v_nlc, cost, path):
    total_mins = int(round(cost))
    hours, mins = divmod(total_mins, 60)
    journey_time_str = f"{hours}:{mins:02d}"

    u_mapped = NLC_ALIASES.get(u_nlc, u_nlc)
    v_mapped = NLC_ALIASES.get(v_nlc, v_nlc)

    from_station = nlc_to_name.get(u_mapped, f"NLC_{u_mapped}")
    to_station = nlc_to_name.get(v_mapped, f"NLC_{v_mapped}")

    ic_nlc, ic_name, all_ics = get_route_interchanges_and_dlr_point(
        path, nlc_to_name
    )

    results.append({
        "from station": from_station,
        "from station NLC": u_mapped,
        "to station": to_station,
        "to station NLC": v_mapped,
        "dlr interchange": ic_name,
        "dlr interchange NLC": ic_nlc,
        "all interchange stations": all_ics,
        "journey time": journey_time_str,
        "Journey_Time_Mins": total_mins,
        "Direction": direction,
    })

  # 1. Non-DLR -> DLR Journeys
  for u_nlc in non_dlr_nlcs:
    u_nodes = [n for n in G.nodes if n.startswith(f"{u_nlc}_")]
    if not u_nodes:
      continue
    for v_nlc in dlr_nlcs:
      v_nodes = [n for n in G.nodes if n.startswith(f"{v_nlc}_DLR")]
      if not v_nodes:
        continue

      best_cost, best_path = float("inf"), None
      for s in u_nodes:
        for t in v_nodes:
          if nx.has_path(G, s, t):
            c = nx.shortest_path_length(G, s, t, weight="weight")
            if c < best_cost:
              best_cost, best_path = c, nx.shortest_path(
                  G, s, t, weight="weight"
              )

      if best_path:
        add_route_record(
            "Non-DLR to DLR", u_nlc, v_nlc, best_cost, best_path
        )

  # 2. DLR -> Non-DLR Journeys
  for u_nlc in dlr_nlcs:
    u_nodes = [n for n in G.nodes if n.startswith(f"{u_nlc}_DLR")]
    if not u_nodes:
      continue
    for v_nlc in non_dlr_nlcs:
      v_nodes = [n for n in G.nodes if n.startswith(f"{v_nlc}_")]
      if not v_nodes:
        continue

      best_cost, best_path = float("inf"), None
      for s in u_nodes:
        for t in v_nodes:
          if nx.has_path(G, s, t):
            c = nx.shortest_path_length(G, s, t, weight="weight")
            if c < best_cost:
              best_cost, best_path = c, nx.shortest_path(
                  G, s, t, weight="weight"
              )

      if best_path:
        add_route_record(
            "DLR to Non-DLR", u_nlc, v_nlc, best_cost, best_path
        )

  # Format output DataFrame
  df_out = pd.DataFrame(results).drop_duplicates(
      subset=["from station NLC", "to station NLC"]
  )

  cols = [
      "from station",
      "from station NLC",
      "to station",
      "to station NLC",
      "dlr interchange",
      "dlr interchange NLC",
      "all interchange stations",
      "journey time",
  ]
  master_routes_df = df_out[cols]

  master_routes_df.to_csv("dlr_journey_interchanges.csv", index=False)
  print(
      "\nRouting Complete: Successfully generated"
      f" {len(master_routes_df):,} total route pairs in"
      " 'dlr_journey_interchanges.csv'."
  )
  return master_routes_df

master_routes_df = run_batch_dlr_routing()
master_routes_df.head()

In [ ]:
# ==============================================================================
# DLR ROUTING AUDIT & COMPARISON SCRIPT
# ==============================================================================
# Compares 'dlr_journey_interchanges.csv' against the manual review baseline.
# ==============================================================================

import os
import pandas as pd

# 1. Load datasets
manual_file = "DLR interchange manual review.csv"
auto_file = "dlr_journey_interchanges.csv"

if not os.path.exists(manual_file) or not os.path.exists(auto_file):
  print(
      "Error: Required CSV files not found. Ensure both files are in the"
      " directory."
  )
else:
  df_manual = pd.read_csv(manual_file)
  df_auto = pd.read_csv(auto_file)

  # Standardize column names
  df_manual.columns = df_manual.columns.str.strip()
  df_auto.columns = df_auto.columns.str.strip()

  # Master NLC Alias Unification Map (to bridge old/new and adjacent stations)
  NLC_ALIASES = {
      843: 884,  # Canning Town
      573: 5152,  # Woolwich Arsenal
      844: 513,  # Bank
      863: 719,  # Stratford
      6960: 719,  # Stratford NR
      1082: 860,  # Shadwell LO
      842: 852,  # Canary Wharf
      887: 6561,  # Custom House
      928: 5145,  # Greenwich
      862: 7491,  # Limehouse
      940: 5051,  # Lewisham
      559: 889,  # West Silvertown
      528: 851,  # Bow Road LU -> Bow Church DLR
      644: 513,  # Monument LU -> Bank Unified Hub
      731: 888,  # Tower Hill LU -> Tower Gateway DLR
  }

  # Apply NLC mapping
  df_manual["NLC1_mapped"] = (
      pd.to_numeric(df_manual["NLC1"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )
  df_manual["NLC2_mapped"] = (
      pd.to_numeric(df_manual["NLC2"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )

  df_auto["from_NLC_mapped"] = (
      pd.to_numeric(df_auto["from station NLC"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )
  df_auto["to_NLC_mapped"] = (
      pd.to_numeric(df_auto["to station NLC"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )

  # Deduplicate
  m_uniq = df_manual.drop_duplicates(subset=["NLC1_mapped", "NLC2_mapped"]).copy()
  a_uniq = df_auto.drop_duplicates(
      subset=["from_NLC_mapped", "to_NLC_mapped"]
  ).copy()

  # Metric 1: Total number of routes and unique NLC stations
  print("=== 1. Total Routes & Unique NLC Stations ===")
  print(f"Total Unique Routes in Manual Review: {len(m_uniq):,}")
  print(f"Total Unique Routes in Automated Graph: {len(a_uniq):,}")

  manual_nlcs = set(m_uniq["NLC1_mapped"]).union(set(m_uniq["NLC2_mapped"]))
  auto_nlcs = set(a_uniq["from_NLC_mapped"]).union(
      set(a_uniq["to_NLC_mapped"])
  )
  print(f"Unique NLC Gate Lines in Manual Review: {len(manual_nlcs):,}")
  print(f"Unique NLC Gate Lines in Automated Graph: {len(auto_nlcs):,}")

  # Metric 2: Distribution of DLR Interchanges Found
  print("\n=== 2. Distribution of DLR Interchanges Found (Top Hubs) ===")
  print("\nManual Review DLR Interchanges:")
  print(m_uniq["DLR_INT Name"].value_counts().head(10))

  print("\nAutomated Graph DLR Interchanges:")
  print(a_uniq["dlr interchange"].value_counts().head(10))

  # Metric 3: Routes with no interchanges vs interchanges present
  m_no_ic = m_uniq["DLR_INT"].isna() | (m_uniq["DLR_INT"] == 0)
  a_no_ic = (
      a_uniq["dlr interchange"].isna()
      | (a_uniq["dlr interchange"] == "Direct / None")
      | (a_uniq["dlr interchange"] == "")
  )

  print("\n=== 3. Interchange Presence Comparison ===")
  print(f"Manual Routes with No DLR Interchange Logged: {m_no_ic.sum():,}")
  print(f"Automated Routes with No DLR Interchange Found: {a_no_ic.sum():,}")

  # Merge datasets for direct overlap analysis
  merged = pd.merge(
      m_uniq,
      a_uniq,
      left_on=["NLC1_mapped", "NLC2_mapped"],
      right_on=["from_NLC_mapped", "to_NLC_mapped"],
      how="outer",
      indicator=True,
  )

  print("\n=== 4. Dataset Overlap & Agreement ===")
  merge_counts = merged["_merge"].value_counts()
  print(f"Shared O-D Pairs (Matched Both): {merge_counts.get('both', 0):,}")
  print(f"Manual Review Only: {merge_counts.get('left_only', 0):,}")
  print(f"Automated Graph Only: {merge_counts.get('right_only', 0):,}")

In [ ]:
# Diagnostic: Find NLCs present in manual review but missing from automated graph
manual_nlcs = set(df_manual['NLC1']).union(set(df_manual['NLC2']))
auto_nlcs = set(df_auto['from station NLC']).union(
    set(df_auto['to station NLC'])
)

missing_nlcs = manual_nlcs - auto_nlcs
print(f'Number of NLCs in manual review missing from automated graph: {len(missing_nlcs)}')

In [ ]:
# ==============================================================================
# DIAGNOSTIC: FIND STATIONS IN all_stations.csv MISSING FROM line_travel_times.csv
# ==============================================================================

# 1. Load datasets
df_stations = pd.read_csv("all_stations.csv")
df_travel = pd.read_csv("line_travel_times.csv")

# 2. Extract unique NLC sets
all_station_nlcs = set(
    pd.to_numeric(df_stations["NLC"], errors="coerce").dropna().astype(int)
)
travel_nlcs = set(
    pd.to_numeric(df_travel["From_NLC"], errors="coerce").dropna().astype(int)
).union(
    set(pd.to_numeric(df_travel["To_NLC"], errors="coerce").dropna().astype(int))
)

# 3. Identify missing NLCs
missing_nlcs = all_station_nlcs - travel_nlcs
df_missing = df_stations[
    df_stations["NLC"].astype(int).isin(missing_nlcs)
].copy()
df_missing.to_csv("stations_missing_from_travel_times.csv", index=False)

# 4. Print summary metrics
print(f"Total stations in all_stations.csv: {len(df_stations):,}")
print(
    "Stations with valid travel time edges:"
    f" {len(all_station_nlcs) - len(missing_nlcs):,}"
)
print(f"Stations missing from travel times: {len(df_missing):,}\n")

# Display missing stations grouped by transport operator/joint network
if not df_missing.empty:
  print("Breakdown of missing stations by network joint type:")
  print(df_missing["JOINT"].value_counts())
  print("\nFirst 15 missing stations:")
  display(df_missing[["NLC", "STATION", "JOINT", "GATED"]])
else:
  print("All stations in all_stations.csv have travel time edges!")